# Qwen3-VL-8B -- Climate generalization pilot

Content-domain + chart-type generalization test (Chapter 7 Limitation 2, Chapter 8 item 4,
supervisor item 7). Solar-vs-wind line chart, fabricated data, climate-framed claim -- see
`climate_pilot/generate_climate_stimuli.py` for the stimulus design rationale. Same protocols
as the main study (baseline single-image like/scroll + logprobs, single-image across the 6
`metrics/realistic` engagement scales + logprobs, full 7x7 paired A/B `metrics` grid), pointed
at `climate_pilot/posts/` instead of `benchmarking/`. 25 posts (not 50/100) -- this is a scoped
pilot, not a full replication.

In [ ]:
import sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

print("✅ Done — restart the kernel now")

Restart kernel after running the setup cell above.

In [ ]:
!nvidia-smi

In [ ]:
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("✅ Loaded successfully")

In [ ]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import (
    LIKE_PROMPT_SINGLE, LIKE_PROMPT_YESNO, LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_baseline, run_e1_metrics, run_e1_metrics_paired
)
from e1_utils.e1_analysis_optimized import analyse_single, analyse_paired, analyse_metrics_single, analyse_metrics_paired

# --- Configuration: climate pilot, NOT the main benchmarking/ pool ---
EXPERIMENT_DIR = Path().resolve().parent        # experiments/e1_climate/  -- shared across all 4 climate-pilot models, so they see the identical 25-image sample
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 25

correct_dir = ROOT_DIR / "climate_pilot/posts/correct/PNGs"
incorrect_dir = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs"
all_images = build_paired_sample(correct_dir, incorrect_dir, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# metrics/realistic condition only -- the condition that showed the strongest conformity effect
# in the main study (Section 6.2)
correct_base = ROOT_DIR / "climate_pilot/posts/correct/PNGs/metrics/realistic"
incorrect_base = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs/metrics/realistic"


In [ ]:
from e1_utils.inference_qwen import run_inference_qwen

In [ ]:
from e1_utils.e1_optimized import (
    run_e1_baseline_logprobs, run_e1_metrics_logprobs,
    LIKE_CANDIDATES_SINGLE, LIKE_CANDIDATES_YESNO
)

from e1_utils.inference_qwen import run_inference_with_scores_qwen

## Approach 1 -- single image, like/scroll, baseline (0 engagement)

In [ ]:
run_e1_baseline(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_baseline.json",
              inference_fn=run_inference_qwen)

In [ ]:
run_e1_baseline_logprobs(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_baseline_logprobs.json", score_fn=run_inference_with_scores_qwen)

In [ ]:
analyse_single(OUTPUT_DIR, "e1_results_baseline.json", like_answer="like")

## Approach 1 variant -- single image, like/scroll, across the 6 `metrics/realistic` engagement scales

In [ ]:
run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_metrics.json",
              inference_fn=run_inference_qwen)

In [ ]:
run_e1_metrics_logprobs(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_metrics_logprobs.json", score_fn=run_inference_with_scores_qwen)

In [ ]:
analyse_metrics_single(OUTPUT_DIR, "e1_results_metrics.json", like_answer="like")

## Approach 2 -- paired A/B forced choice, full 7x7 `metrics/realistic` disparity grid

In [ ]:
import time
start = time.time()
run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
              prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_paired.json",
              inference_fn=run_inference_qwen)
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min")

In [ ]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_metrics_paired.json")